# Assignment 2: Animal Classification Using CNN

**Type:** Multi-Class Classification  
**Dataset:** Animal Image Classification – 5 Species

**Student Name:** ____________________  
**Roll Number:** ____________________

## Introduction
In this assignment, a basic Convolutional Neural Network (CNN) is used to classify animal images into five classes. The notebook covers dataset exploration, preprocessing, data augmentation, CNN training, evaluation, confusion matrix, classification report, and sample predictions.

## 1. Objectives
- Explore the animal image dataset.
- Prepare images for CNN.
- Split data into training, validation, and testing sets.
- Build a simple CNN from scratch.
- Train and evaluate the model.
- Analyze predictions using standard classification metrics.

## 2. Import Libraries

In [ ]:
import os
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)

## 3. Download Dataset

The Kaggle dataset is downloaded directly. In a cloud environment, make sure Kaggle API access is configured first.

In [ ]:
!pip -q install kaggle
!mkdir -p ~/.kaggle

# Upload kaggle.json to /content before running this cell if required.
if os.path.exists('/content/kaggle.json'):
    shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d miadul/animal-image-classification-5-species -p /content/animal_dataset --unzip
print('Dataset downloaded.')

## 4. Locate Dataset and Classes

In [ ]:
BASE_DIR = Path('/content/animal_dataset')
EXTS = {'.jpg','.jpeg','.png','.bmp','.webp'}


def count_images(folder):
    return sum(1 for p in Path(folder).rglob('*') if p.is_file() and p.suffix.lower() in EXTS)

# The downloaded Kaggle dataset already contains train/test folders.
# We must NOT select the test folder as the main dataset.
DATASET_ROOT = BASE_DIR / 'animals_dataset'

if not DATASET_ROOT.exists():
    possible = [p for p in BASE_DIR.rglob('*') if p.is_dir() and (p / 'train').is_dir() and (p / 'test').is_dir()]
    if possible:
        DATASET_ROOT = possible[0]

TRAIN_DIR = DATASET_ROOT / 'train'
TEST_DIR = DATASET_ROOT / 'test'
VAL_DIR = DATASET_ROOT / 'validation'

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError('Could not find the original train and test folders in the downloaded dataset.')

print('Dataset root:', DATASET_ROOT)
print('Train folder:', TRAIN_DIR)
print('Test folder:', TEST_DIR)
print('Validation folder exists:', VAL_DIR.exists())

class_folders = sorted([p for p in TRAIN_DIR.iterdir() if p.is_dir() and count_images(p) > 0])
class_names = [p.name for p in class_folders]

print('Classes:', class_names)
print('Number of classes:', len(class_names))
print('Train images:', count_images(TRAIN_DIR))
print('Test images:', count_images(TEST_DIR))
if VAL_DIR.exists():
    print('Validation images:', count_images(VAL_DIR))


## 5. Dataset Exploration

In [ ]:
counts = {p.name: count_images(p) for p in class_folders}
df = pd.DataFrame({'Animal Class': list(counts), 'Training Images': list(counts.values())})
display(df)
print('Total training images:', df['Training Images'].sum())

plt.figure(figsize=(9,5))
sns.barplot(data=df, x='Animal Class', y='Training Images')
plt.title('Number of Training Images per Class')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 6. Sample Images

In [ ]:
random.seed(42)
plt.figure(figsize=(15,10))
plot_no = 1
for folder in class_folders:
    files = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
    for f in random.sample(files, min(3, len(files))):
        plt.subplot(len(class_folders), 3, plot_no)
        plt.imshow(plt.imread(f))
        plt.title(folder.name)
        plt.axis('off')
        plot_no += 1
plt.tight_layout()
plt.show()


## 7. Train / Validation / Test Split

The Kaggle dataset already provides a separate training and test folder. The original test set is kept untouched. If a validation folder is not provided, 20% of the training data is used for validation.


In [ ]:
# We keep the original Kaggle test set untouched.
# Validation is taken from the training folder only when needed.

if VAL_DIR.exists():
    print('Using the validation folder provided by the dataset.')
    print('Train images:', count_images(TRAIN_DIR))
    print('Validation images:', count_images(VAL_DIR))
    print('Test images:', count_images(TEST_DIR))
else:
    print('No separate validation folder found.')
    print('A 20% validation split will be created from TRAIN data only.')
    print('The original TEST data will remain completely unseen until evaluation.')


## 8. Image Preprocessing and Augmentation

Images are resized to 128×128 and normalized to 0–1. Simple augmentation is applied only to training images.

In [ ]:
IMG_SIZE = (128,128)
BATCH_SIZE = 32

if VAL_DIR.exists():
    train_gen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        horizontal_flip=True,
        zoom_range=0.10
    )
    valid_gen = ImageDataGenerator(rescale=1./255)

    train_data = train_gen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=True, seed=42
    )
    val_data = valid_gen.flow_from_directory(
        VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False
    )
else:
    train_gen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        horizontal_flip=True,
        zoom_range=0.10,
        validation_split=0.20
    )
    valid_gen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.20
    )

    train_data = train_gen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='training', shuffle=True, seed=42
    )
    val_data = valid_gen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', subset='validation', shuffle=False, seed=42
    )

test_gen = ImageDataGenerator(rescale=1./255)
test_data = test_gen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print('Class mapping:', train_data.class_indices)
print('Training samples:', train_data.samples)
print('Validation samples:', val_data.samples)
print('Test samples:', test_data.samples)


## 9. Build Basic CNN

In [ ]:
num_classes = len(class_names)

model = models.Sequential([
    layers.Input(shape=(128,128,3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()


## 10. Compile Model

In [ ]:
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

early_stop=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)
checkpoint=tf.keras.callbacks.ModelCheckpoint('/content/best_animal_cnn.keras',monitor='val_loss',save_best_only=True)

## 11. Train CNN

In [ ]:
history=model.fit(train_data,validation_data=val_data,epochs=15,callbacks=[early_stop,checkpoint])

## 12. Accuracy and Loss Graphs

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'],label='Training Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')
plt.title('Training vs Validation Accuracy'); plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(); plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history['loss'],label='Training Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')
plt.title('Training vs Validation Loss'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(); plt.show()

## 13. Test Evaluation

In [ ]:
test_data.reset()
test_loss,test_accuracy=model.evaluate(test_data,verbose=1)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy*100:.2f}%')

## 14. Predictions and Confusion Matrix

In [ ]:
test_data.reset()
probs=model.predict(test_data)
y_pred=np.argmax(probs,axis=1)
y_true=test_data.classes
idx_to_class={v:k for k,v in test_data.class_indices.items()}
labels=[idx_to_class[i] for i in range(num_classes)]

cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=labels,yticklabels=labels)
plt.title('Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout(); plt.show()

## 15. Classification Report

In [ ]:
print(classification_report(y_true,y_pred,target_names=labels,digits=4))

## 16. Sample Test Predictions

In [ ]:
test_data.reset()
images,labels_onehot=next(test_data)
probs=model.predict(images,verbose=0)
preds=np.argmax(probs,axis=1)
actual=np.argmax(labels_onehot,axis=1)

plt.figure(figsize=(15,10))
for i in range(min(10,len(images))):
    plt.subplot(2,5,i+1); plt.imshow(images[i]); plt.axis('off')
    conf=probs[i,preds[i]]*100
    plt.title(f'Actual: {idx_to_class[actual[i]]}\nPred: {idx_to_class[preds[i]]} ({conf:.1f}%)',fontsize=9)
plt.tight_layout(); plt.show()

## 17. Wrong Predictions

In [ ]:
wrong=np.where(y_true!=y_pred)[0]
print('Wrong predictions:',len(wrong))
if len(wrong)>0:
    test_data.reset(); all_x=[]; all_y=[]
    for x,y in test_data:
        all_x.append(x); all_y.append(y)
        if sum(len(a) for a in all_x)>=test_data.samples: break
    all_x=np.concatenate(all_x)[:test_data.samples]
    all_y=np.concatenate(all_y)[:test_data.samples]
    p=model.predict(all_x,batch_size=BATCH_SIZE,verbose=0); pp=np.argmax(p,axis=1); aa=np.argmax(all_y,axis=1)
    plt.figure(figsize=(15,8))
    for j,i in enumerate(np.where(aa!=pp)[0][:6]):
        plt.subplot(2,3,j+1); plt.imshow(all_x[i]); plt.axis('off')
        plt.title(f'Actual: {idx_to_class[aa[i]]}\nPredicted: {idx_to_class[pp[i]]}')
    plt.tight_layout(); plt.show()
else:
    print('No wrong predictions found in the test set.')

## 18. Save Model

In [ ]:
model.save('/content/animal_classification_cnn.keras')
print('Saved: /content/animal_classification_cnn.keras')

## 19. Final Results and Conclusion

The CNN was trained from scratch to classify the five animal classes. The final performance should be reported using the **actual Test Accuracy** printed above. The confusion matrix and classification report show which classes were easier or harder for the model to recognize.

### Conclusion
This assignment demonstrated the complete basic workflow of an image classification problem: dataset exploration, preprocessing, augmentation, CNN training, evaluation, and prediction. The model can be improved in future work by using more training data, tuning the CNN architecture, or applying transfer learning.